# Manuscript

Render it to markdown and pdf

:warning: TODO: Use a different writer for each medium type

In [6]:
# Can Render to a PDF using pandoc
!sudo apt -y install pandoc
%pip install markdown2 pypandoc

# If using pdflatex
!sudo apt -y install texlive texlive-latex-extra

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pandoc is already the newest version (2.9.2.1-3ubuntu2).
0 upgraded, 0 newly installed, 0 to remove and 214 not upgraded.
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
texlive is already the newest version (2021.20220204-1).
texlive-latex-extra is already the newest version (2021.20220204-1).
0 upgraded, 0 newly installed, 0 to remove and 214 not upgraded.


# Convert to Markdown and LaTeX

## Load the manuscript

In [7]:
import settings
from model import Story

story = Story.load_from_directory(settings.STORY_DIR + "/step_14")

In [8]:
story.get_story_dialogue()

StoryDialogue(act_dialogues=[ActDialogue(act_id='unraveling_shadows', chapter_dialogues=[ChapterDialogue(chapter_id='CH01-US', scene_dialogues=[SceneDialogue(scene_id='SC01-CH01-US', dialogues=[], content="\n\n**Scene: A Typical Session**\n**Scene ID: SC01-CH01-US**\nEliza, with her tailored suit pressed to perfection and a serene expression that belies the storm within, stands behind her mahogany desk in her meticulously organized office at the psychiatric hospital. The room is bathed in soft natural light filtering through sheer curtains, casting gentle shadows on the walls lined with books--a testament to Eliza's love for knowledge and understanding of human psyche.\n\nPatient #1 sits across from her, a man whose eyes are heavy with unshed tears but who listens intently as she speaks in measured tones that convey both empathy and authority--a professional at the peak of his craft. The air is thick with anticipation; Eliza knows this session could be pivotal for him or her, depending

In [9]:
def story_to_markdown(story):
    """
    Converts a Story and its associated StoryDialogue to a Markdown manuscript, formatted like a novel with chapter pages.
    """
    # Start with the title page
    markdown_content = f"# {story.title}\n\n"

    # Loop over each act and chapter to create the story content
    for act_index, (act, act_dialog) in enumerate(zip(story.acts, story.get_story_dialogue().act_dialogues), start=1):
        # Add a title page for each act
        markdown_content += f"\n\n# Act {act_index}: {act.title}\n\n"
        markdown_content += f"_{act.description}_\n\n" if act.description else ""
        markdown_content += "<div style='page-break-after: always;'></div>\n\n"  # Page break for act separation

        # Loop over each chapter in the act
        for chapter_index, (chapter, chapter_dialog) in enumerate(zip(act.chapters, act_dialog.chapter_dialogues), start=1):
            # Add a chapter title page
            markdown_content += f"## Chapter {chapter_index}: {chapter.title}\n\n"
            markdown_content += f"_{chapter.description}_\n\n" if chapter.description else ""

            # Loop over each scene in the chapter
            for scene_index, (scene, scene_dialog) in enumerate(zip(chapter.scenes, chapter_dialog.scene_dialogues), start=1):
                markdown_content += f"\n\n---\n\n"  # Separator for scenes

                # Optional: Add scene title and description
                markdown_content += f"### Scene {scene_index}: {scene.title}\n\n"
                markdown_content += f"_{scene.description}_\n\n" if scene.description else ""

                # Add each dialogue line for the scene
                for dialogue_line in scene_dialog.dialogues:
                    # Dialogue formatted as prose
                    markdown_content += f"{dialogue_line.character_nickname}: {dialogue_line.line}\n\n"

    return markdown_content


In [10]:
# Render the markdown
markdown_content = story_to_markdown(story)

In [11]:
# Save it
import os
os.makedirs(f"{settings.STORY_DIR}/step_14", exist_ok=True)
manuscript_md_path = f"{settings.STORY_DIR}/step_14/manuscript.md"

# Save to a Markdown file
with open(manuscript_md_path, "w") as file:
    file.write(markdown_content)

In [12]:
import markdown2
import pypandoc

def markdown_to_latex(markdown_text):
    """
    Convert markdown text to LaTeX using markdown2 and pypandoc.
    """
    # Convert markdown to HTML first
    html_text = markdown2.markdown(markdown_text)
    # Convert HTML to LaTeX
    latex_text = pypandoc.convert_text(html_text, 'latex', format='html')
    return latex_text

In [ ]:
def story_to_latex_graphic_novel(story, scene_image_dir):
    """
    Converts a Story and its associated StoryDialogue to a LaTeX manuscript formatted as a graphic novel,
    where each chapter contains multiple scenes, and images may be included.
    """

    # Start with document preamble
    latex_content = r"""
    \documentclass[12pt]{report}  % Using 'report' class to avoid blank pages between chapters
    \usepackage{setspace}
    \usepackage{titlesec}
    \usepackage{graphicx}
    \usepackage{url}
    \titleformat{\chapter}[display]
      {\normalfont\huge\bfseries}{\chaptername\ \thechapter}{20pt}{\Huge}
    \usepackage{fancyhdr}
    \pagestyle{fancy}
    \fancyhf{}
    \rhead{\thepage}
    \begin{document}
    \onehalfspacing
    \pagenumbering{gobble} % Suppress page numbers until Chapter 1
    """

    # Title page with title image if available, without a page break
    title_image_path = os.path.join(scene_image_dir, "cover.png")
    latex_content += r"\begin{center}\n"
    
    # Include the title image if available
    if os.path.exists(title_image_path):
        latex_content += f"\\includegraphics[width=0.8\\textwidth]{{{title_image_path}}}\n\\vspace{{1cm}}\n"
    
    # Add the title text below the image
    latex_content += r"""
    {\Huge \textbf{""" + story.title + r"""}} \\[2cm]
    \end{center}
    """

    # Start page numbering with Chapter 1
    latex_content += r"""
    \newpage
    \pagenumbering{arabic} % Start page numbering
    """

    # Loop over each act, chapter, and scene to create the story content
    chapter_counter = 1
    for act_idx, (act, act_dialogue) in enumerate(zip(story.acts, story.get_story_dialogue().act_dialogues), start=1):
        # Add act title
        latex_content += f"\\chapter*{{Act {act_idx}: {act.title}}}\n"
        latex_content += f"\\addcontentsline{{toc}}{{chapter}}{{Act {act_idx}: {act.title}}}\n"
        if act.description:
            latex_content += f"\\textit{{{act.description}}}\n\n"

        # Loop over chapters in the act
        for chapter_idx, (chapter, chapter_dialogue) in enumerate(zip(act.chapters, act_dialogue.chapter_dialogues), start=1):
            # Add chapter title
            latex_content += f"\\section*{{Chapter {chapter_counter}: {chapter.title}}}\n"
            latex_content += f"\\addcontentsline{{toc}}{{section}}{{Chapter {chapter_counter}: {chapter.title}}}\n"
            chapter_counter += 1
            if chapter.description:
                latex_content += f"\\textit{{{chapter.description}}}\n\n"

            # Loop over scenes in the chapter
            for scene_idx, (scene, scene_dialogue) in enumerate(zip(chapter.scenes, chapter_dialogue.scene_dialogues), start=1):
                # Add scene title
                latex_content += f"\\subsection*{{Scene {scene_idx}: {scene.title}}}\n"
                if scene.description:
                    latex_content += f"\\textit{{{scene.description}}}\n\n"

                # Include scene image if it exists
                scene_image_path = os.path.join(scene_image_dir, f"{scene.scene_id}.live.png")
                if os.path.exists(scene_image_path):
                    latex_content += f"\\begin{{center}}\n\\includegraphics[width=0.8\\textwidth]{{{scene_image_path}}}\n\\end{{center}}\n\n"

                # Use the content field for the full scene dialogue and actions, converting from markdown to latex
                if scene_dialogue.content:
                    latex_scene_content = markdown_to_latex(scene_dialogue.content.strip())
                    latex_content += f"{latex_scene_content}\n\n"

    # Add "Made with Plotomatic" statement at the end
    latex_content += r"""
    \newpage
    \vfill
    \begin{center}
    \textit{Made with \textbf{Plotomatic}.} \\
    For more information, visit: \url{https://github.com/mattwilliamson/Plotomatic}
    \end{center}
    """

    # End the document
    latex_content += "\\end{document}"
    
    return latex_content


In [14]:
# latex_content = story_to_latex_novel(story)
latex_content = story_to_latex_graphic_novel(story, settings.STORY_DIR + "/step_6/scenes")
latex_content

AttributeError: 'Act' object has no attribute 'scenes'

In [ ]:
from IPython.display import display, Markdown

step_path = f"{settings.STORY_DIR}/step_14"
manuscript_json_path = f"{step_path}/manuscript.json"
manuscript_latex_path = f"{step_path}/manuscript.tex"
manuscript_pdf_path = f"{step_path}/manuscript.pdf"

# Save to a LaTeX file
with open(manuscript_latex_path, "w") as file:
    file.write(latex_content)

# !pandoc {manuscript_latex_path} -o {manuscript_pdf_path}
!ls -lah {step_path}
!rm manuscript.log
!pdflatex -interaction=nonstopmode -output-directory={step_path} {manuscript_latex_path}

total 1.7M
drwxrwxr-x 2 matt matt 4.0K Nov 18 13:37 .
drwxrwxr-x 8 matt matt 4.0K Nov 16 16:22 ..
-rw-rw-r-- 1 matt matt  133 Nov 18 13:37 manuscript.aux
-rw-rw-r-- 1 matt matt  11K Nov 18 13:37 manuscript.log
-rw-rw-r-- 1 matt matt  102 Nov 18 13:37 manuscript.md
-rw-rw-r-- 1 matt matt 1.4M Nov 18 13:37 manuscript.pdf
-rw-rw-r-- 1 matt matt  20K Nov 18 13:37 manuscript.tex
-rw-rw-r-- 1 matt matt 223K Nov 18 10:49 story_dialog.2024-11-18.json
-rw-rw-r-- 1 matt matt  20K Nov 18 13:33 story_dialog.json
-rw-rw-r-- 1 matt matt  59K Nov 18 13:33 story.json
rm: cannot remove 'manuscript.log': No such file or directory
This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(./stories/my_story/step_14/manuscript.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21>
(/usr/share/texlive/texmf-dist/tex/latex/base/report.cls
Document Class: report 2021/10/04 v1.4n Standard LaTeX 

In [ ]:
display(Markdown(f"# :open_book: [Download the manuscript PDF]({manuscript_pdf_path})"))

# :open_book: [Download the manuscript PDF](stories/my_story/step_14/manuscript.pdf)

# Generate Dialog (for screenplays)
